# ML Challenge

## Contexto

- Predecir los BPM de una canción a partir de características musicales. 
Permite comparar modelos de regresión y evaluar el error mediante RMSE 
y análisis de residuos.

- No es un problema de clasificación, sino de regresión, porque queremos predecir un valor numérico (los BPM).

Tipo: Supervisado · Regresión


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
import matplotlib.pyplot as plt
import seaborn as sns

## 1. Carga de los datos

In [22]:
#Carga de los datos y creación de la copia de seguridad del dataset
data = pd.read_csv("../data/Beats_PM/train.csv")
data_test = pd.read_csv("../data/Beats_PM/test.csv")

data_set_original = data.copy()
data_set_test_original = data_test.copy()

## 2. Exploración y análisis de datos (EDA)

#### Calidad y validación de los datos de entrenamiento

In [ ]:
#Verificación de datos falatantes y nulos
vacio = data.isna().sum()
nulo = data.isnull().sum()
print(f"La cantidad de líneas vacías por columna es: \n{vacio}")
print(f"\nLa cantidad de valores nulos es: \n{nulo}")

#Verificación de filas duplicadas
print("\nNúmero de filas duplicadas:", data.duplicated().sum())

#Evaluación estadística del dataset
print("\nInformación del dataset:\n")
print(data.info())
print("\nDescripción estadística del dataset:\n")
print(data.describe())

#Forma del dataset
filas, columnas = data.shape
print(f"\nForma del Dataset\nFilas: {filas}\nColumnas: {columnas}")

# Condición: La energía, los beats por minuto y la duración de la canción no deben tener valores negativos.
#Así validamos que los datos no generen inconsistencias.
filtro_energy = (data["Energy"] < 0)
filtro_beat = (data["BeatsPerMinute"] < 0)
filtro_duracion = (data["TrackDurationMs"] <= 0)

print("Filas que contienen valores negativos de energía o de beats")
display(data[filtro_energy])
display(data[filtro_beat])
display(data[filtro_duracion])

#### Detección de valores atípicos (método IQR)

In [ ]:
numeric_columns = data.select_dtypes(include='number').columns

for column in numeric_columns:
    Q1 = data[column].quantile(0.25)
    Q3 = data[column].quantile(0.75)
    IQR = Q3 - Q1

    lower_limit = Q1 - 1.5 * IQR
    upper_limit = Q3 + 1.5 * IQR

    outliers = data[
        (data[column] < lower_limit) |
        (data[column] > upper_limit)
    ]

    print(f"{column}: {len(outliers)} outliers")

#### Correlación entre variables

In [ ]:
plt.figure(figsize=(8, 10))
sns.heatmap(data.corr(), annot=True, fmt='.2f', cmap='RdBu_r')
plt.title('Matriz de correlación')
plt.show()

#### Evaluación inicial de los datos

Con la evaluación de los datos se puede tener certeza de su uso, ya que, la validaciones y los rangos se ajustan a la necesidad.

Hay columnas que se evidencia que los valores están normalizados, estas columnas son: "AcousticQuality", "InstrumentalScore" y "Energy"

Hay columnas con valores que se pueden tomar como outliners pero, como los valores ya fueron tratados se pueden iniciar las pruebas para verificar el funcionamiento del algoritmo de entrenamiento.

a continuación la verificación de los datos del dataset de test que proporcionan.

#### Comprobación del conjunto de test

In [ ]:
#Verificación de datos falatantes y nulos
vacio = data_test.isna().sum()
nulo = data_test.isnull().sum()
print(f"La cantidad de líneas vacías por columna es: \n{vacio}")
print(f"\nLa cantidad de valores nulos es: \n{nulo}")

#Verificación de filas duplicadas
print("\nNúmero de filas duplicadas:", data_test.duplicated().sum())

#Evaluación estadística del dataset
print("\nInformación del dataset:\n")
print(data_test.info())
print("\nDescripción estadística del dataset:\n")
print(data_test.describe())

#Forma del dataset
filas, columnas = data_test.shape
print(f"\nForma del Dataset\nFilas: {filas}\nColumnas: {columnas}")

# Condición: La energía, los beats por minuto y la duración de la canción no deben tener valores negativos.
#Así validamos que los datos no generen inconsistencias.
filtro_energy = (data_test["Energy"] < 0)
filtro_beat = (data_test["BeatsPerMinute"] < 0)
filtro_duracion = (data_test["TrackDurationMs"] <= 0)

print("Filas que contienen valores negativos de energía o de beats")
display(data_test[filtro_energy])
display(data_test[filtro_beat])
display(data_test[filtro_duracion])

Al realizar las mismas validaciones de los datos que se realizaron en el conjunto de entrenamiento, se obtuvieron los mismos resultados, por esta razón el conjuto de datos de test también estaría listo para partirlo y utilizar una parte en ajustes del algoritmo y la otra en la verificación final del modelo.